# 04 — Continued Pretraining of mT5-small on the Shona corpus

**Run this on Google Colab with a T4 GPU.** Phase 2 of TauraBot.

**What this notebook does:**
1. Mount Google Drive (where the corpus + checkpoints live).
2. Clone the TauraBot repo (or upload code manually).
3. Install training dependencies.
4. Verify GPU + dataset.
5. Run `src/model/pretrain.py` with span-corruption MLM objective.
6. Push the final model to HuggingFace Hub as `{username}/shona-mt5-small`.

**Expected runtime:** ~2–3 hours on T4 free tier for 10,000 steps.

**Training config** (from `configs/pretrain.yaml`):
- Base: `google/mt5-small` (300 MB)
- max_seq_length: 128, batch 8 × grad-accum 4 = effective batch 32
- learning_rate 5e-4, linear warmup 500 steps
- fp16 + gradient checkpointing (fits T4 VRAM)
- Span corruption: 15% noise, mean span length 3
- Save every 1000 steps to Drive → survives Colab session resets

**Before running, do this once locally:**
1. Push the TauraBot folder to GitHub (private or public), OR
2. Zip the repo + upload to your Google Drive at `MyDrive/taurabot/`
3. Upload `data/processed/corpus.txt` to `MyDrive/taurabot/data/processed/corpus.txt` (360 MB)

## 1. Verify Colab GPU

In [ ]:
!nvidia-smi

Expect to see **Tesla T4** with ~15 GB VRAM. If you get a different GPU or CPU runtime, switch via *Runtime → Change runtime type → T4 GPU*.

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set DRIVE_ROOT to wherever you put the taurabot folder in Drive.
DRIVE_ROOT = '/content/drive/MyDrive/taurabot'
!ls $DRIVE_ROOT | head -20

## 3. Pull repo code into the Colab workspace

Two options — pick one:

**A. From GitHub** (preferred — easier to re-run with latest code):
```python
!git clone https://github.com/{YOUR-USERNAME}/taurabot.git /content/taurabot
```

**B. From Drive** (no GitHub needed): just copy the code dir over.

In [ ]:
# Option B — copy from Drive
import shutil, os
if not os.path.exists('/content/taurabot'):
    shutil.copytree(DRIVE_ROOT, '/content/taurabot', ignore=shutil.ignore_patterns(
        'data', 'venv', '__pycache__', '.git', 'checkpoints',
    ))
%cd /content/taurabot
!ls

## 4. Install dependencies

Only Phase 2 training deps — Colab already has torch + CUDA preinstalled.

In [ ]:
!pip install -q -U 'transformers>=4.40,<5.0' 'accelerate>=0.30,<1.0' \
                   'datasets>=2.18,<3.0' 'huggingface_hub>=0.24,<1.0' \
                   'sentencepiece>=0.1.99' 'protobuf>=3.20,<5.0' \
                   'evaluate>=0.4' pyyaml tensorboard
import torch; print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())

## 5. Symlink the corpus + checkpoint dir into Drive

Reading the corpus directly from Drive is slow. Copy locally; write checkpoints back to Drive so they survive a session reset.

In [ ]:
import os, shutil
os.makedirs('/content/taurabot/data/processed', exist_ok=True)
if not os.path.exists('/content/taurabot/data/processed/corpus.txt'):
    print('Copying corpus from Drive (360 MB, takes ~30s)...')
    shutil.copy(f'{DRIVE_ROOT}/data/processed/corpus.txt',
                '/content/taurabot/data/processed/corpus.txt')
!wc -l /content/taurabot/data/processed/corpus.txt

# Point checkpoints/ at Drive (auto-resume + survive session reset)
DRIVE_CKPT = f'{DRIVE_ROOT}/checkpoints/shona-mt5-small'
os.makedirs(DRIVE_CKPT, exist_ok=True)
local_ckpt = '/content/taurabot/checkpoints/shona-mt5-small'
os.makedirs(os.path.dirname(local_ckpt), exist_ok=True)
if not os.path.islink(local_ckpt) and not os.path.exists(local_ckpt):
    os.symlink(DRIVE_CKPT, local_ckpt)
print('Checkpoints will be written to:', DRIVE_CKPT)
!ls $DRIVE_CKPT 2>/dev/null || echo '(no existing checkpoints — fresh run)'

## 6. HuggingFace Hub login (optional, only needed to push the model)

Run this *before* training if you want `push_to_hub: true`.

Get a write-scoped token at <https://huggingface.co/settings/tokens>.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 7. Run pretraining

Drives `src/model/pretrain.py`. The first run will tokenize the corpus (~5 min), cache it, then start training. Subsequent runs reuse the tokenized cache and **auto-resume** from the latest checkpoint in `checkpoints/`.

In [ ]:
# Edit any of these flags if you want to deviate from configs/pretrain.yaml.
# Examples:
#   --max_steps 200            (quick smoke run)
#   --push_to_hub true --hub_model_id YOUR-USERNAME/shona-mt5-small

!cd /content/taurabot && python -m src.model.pretrain \
    --config configs/pretrain.yaml \
    --output_dir checkpoints/shona-mt5-small

## 8. Inspect training logs

TensorBoard reads from `checkpoints/.../runs/` (written by the HF Trainer).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/taurabot/checkpoints/shona-mt5-small

## 9. Sanity-check the trained model with a span infill

Span-corruption-pretrained models can be probed with `<extra_id_0>` masks. If pretraining worked, the model should fill in plausible Shona words.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

ckpt_dir = '/content/taurabot/checkpoints/shona-mt5-small'
tok = AutoTokenizer.from_pretrained(ckpt_dir)
mdl = AutoModelForSeq2SeqLM.from_pretrained(ckpt_dir).to('cuda').eval()

prompts = [
    'Mhoro, ndinodaidzwa kuti <extra_id_0>',          # Hello, my name is ___
    'Pakutanga Mwari akasika <extra_id_0> nenyika',    # In the beginning God created ___ and earth
    'Zita rangu ndiTaura uye ndinotaura <extra_id_0>', # My name is Taura and I speak ___
]
for p in prompts:
    enc = tok(p, return_tensors='pt').to('cuda')
    out = mdl.generate(**enc, max_new_tokens=30, num_beams=4)
    print(f'\nIN:  {p}')
    print(f'OUT: {tok.decode(out[0], skip_special_tokens=False)}')

## 10. Push to HuggingFace Hub (optional)

If you set `--push_to_hub true` and a `--hub_model_id` in step 7, the Trainer already pushed automatically. Otherwise:

In [ ]:
# from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
# mdl.push_to_hub('YOUR-USERNAME/shona-mt5-small')
# tok.push_to_hub('YOUR-USERNAME/shona-mt5-small')
print('See https://huggingface.co/YOUR-USERNAME/shona-mt5-small after pushing.')

## What's next

Phase 3 — conversation fine-tuning on hand-crafted Shona Q&A pairs. See `notebooks/05_conversation_finetuning.ipynb` (to be built).

Don't forget to write a **model card** for the HF Hub model: training data, hyperparameters, intended use, and limitations. A template is included in the README's citation block.